# Supervised Learning

## 1. Setup

Imports libraries and defines constants used throughout the notebook: the data directory, the four prediction checkpoints, and the rolling-average window for pregame features. The `read_data` helper loads CSVs from the shared `data/` folder and keeps `GAME_ID` as a string so its leading zeros are preserved across joins.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data")
CHECKPOINTS = ["_pre-game", "EndQ1", "EndQ2", "EndQ3"]
IN_GAME = ["EndQ1", "EndQ2", "EndQ3"]
ROLLING_WINDOW = 5  # default rolling average file to use
RANDOM_STATE = 42

def read_data(file_name):
    """Read a CSV from the data folder, casting GAME_ID as a string to keep leading zeros."""
    return pd.read_csv(DATA_DIR / f"{file_name}.csv", dtype = {"GAME_ID": str})

## 2. Load Data

Loads the CSVs produced by `nba_stats_data.ipynb` and `kalshi_data.ipynb`: full-game box scores (used to build leak-free rolling features, never as direct inputs), per-checkpoint game tables with Kalshi tickers and outcomes, in-game box stats at each quarter checkpoint, and Kalshi market snapshots. 

In [4]:
full = read_data("_team-box-stats(full-game)")
main = {checkpoint: read_data(f"main({checkpoint})")                    for checkpoint in CHECKPOINTS}
in_game = {checkpoint: read_data(f"in-game-box-stats({checkpoint})")    for checkpoint in IN_GAME}
kalshi = {checkpoint: read_data(f"kalshi({checkpoint})")                for checkpoint in CHECKPOINTS}
rolling_avg = read_data(f"roll-avg-only({ROLLING_WINDOW}GAME)")

**Sanity check: row counts and coverage.** Confirms each checkpoint table covers the expected number of games, how many games have a matching Kalshi market for each team, that the in-game and rolling stats have two team rows per game, and the home-team win rate that serves as the naive baseline.

In [12]:
for checkpoint in CHECKPOINTS:
    m, k = main[checkpoint], kalshi[checkpoint]
    tickers = set(k["KALSHI_ID"])
    print(
        f"{checkpoint:10s} games={m['GAME_ID'].nunique():5d}  "
        f"kalshi_rows={len(k):5d}  "  
        f"home_mkt_found={m['KALSHI_ID_HOME'].isin(tickers).mean():.1%}  "
        f"away_mkt_found={m['KALSHI_ID_AWAY'].isin(tickers).mean():.1%}"
    )
print()

for checkpoint in IN_GAME:
    print(f"in-game {checkpoint}: {len(in_game[checkpoint])} team rows -> {in_game[checkpoint]['GAME_ID'].nunique()} games")
print()

print(f"rolling ({ROLLING_WINDOW}g): {len(rolling_avg)} team rows -> {rolling_avg['GAME_ID'].nunique()} games")

home_win_rate = (main["_pre-game"]["WINNER"] == main["_pre-game"]["NAME_HOME"]).mean()
print(f"home win rate: {home_win_rate:.1%}")

_pre-game  games= 1225  kalshi_rows= 2446  home_mkt_found=99.8%  away_mkt_found=99.9%
EndQ1      games= 1225  kalshi_rows= 2446  home_mkt_found=99.8%  away_mkt_found=99.9%
EndQ2      games= 1225  kalshi_rows= 2443  home_mkt_found=99.6%  away_mkt_found=99.8%
EndQ3      games= 1225  kalshi_rows= 2432  home_mkt_found=98.9%  away_mkt_found=99.6%

in-game EndQ1: 2460 team rows -> 1230 games
in-game EndQ2: 2460 team rows -> 1230 games
in-game EndQ3: 2460 team rows -> 1230 games

rolling (5g): 2340 team rows -> 1173 games
home win rate: 55.4%


**Sanity check: games missing from the checkpoint tables.** Lists games present in the full-season box scores but absent from the Kalshi checkpoint tables. All five are neutral-site games (Mexico City, Las Vegas NBA Cup semifinals, Berlin, London), where home/away designation does not apply.

In [14]:
missing = set(full["GAME_ID"]) - set(main["_pre-game"]["GAME_ID"])

full[full["GAME_ID"].isin(missing)][["GAME_ID", "GAME_DATE", "MATCHUP"]].drop_duplicates("GAME_ID")

,GAME_ID,GAME_DATE,MATCHUP
166,0022500147,2025-11-01,DAL @ DET
744,0022501229,2025-12-13,NYK @ ORL
746,0022501230,2025-12-13,SAS @ OKC
1209,0022500578,2026-01-15,ORL @ MEM
1255,0022500602,2026-01-18,ORL @ MEM


## 3. Exploratory Data Analysis

Before building features, these checks examine whether the Kalshi prices behave like real market probabilities and whether the in-game stats carry the expected signal about the outcome.

In [17]:
def kalshi_home_prob(checkpoint):
    """One row per game: Kalshi bid/ask midpoint for each team's market, plus the outcome."""
    m = main[checkpoint][["GAME_ID", "KALSHI_ID_HOME", "KALSHI_ID_AWAY", "NAME_HOME", "WINNER"]].copy()
    k = kalshi[checkpoint].copy()
    k["MID"] = (k["yes_bid_close"] + k["yes_ask_close"]) / 2
    mid = k.set_index("KALSHI_ID")["MID"]

    m["PROB_HOME"] = m["KALSHI_ID_HOME"].map(mid)
    m["PROB_AWAY"] = m["KALSHI_ID_AWAY"].map(mid)
    m["HOME_WIN"] = (m["WINNER"] == m["NAME_HOME"]).astype(int)
    m["CHECKPOINT"] = checkpoint

    return m

kalshi_long = pd.concat([kalshi_home_prob(checkpoint) for checkpoint in CHECKPOINTS], ignore_index=True)

In [25]:
kalshi_long.head()

,GAME_ID,KALSHI_ID_HOME,KALSHI_ID_AWAY,NAME_HOME,WINNER,PROB_HOME,PROB_AWAY,HOME_WIN,CHECKPOINT,PROB_SUM
0,0022500002,KXNBAGAME-25OCT21GSWLAL-LAL,KXNBAGAME-25OCT21GSWLAL-GSW,Los Angeles Lakers,Golden State Warriors,0.425,0.585,0,_pre-game,1.010
1,0022500001,KXNBAGAME-25OCT21HOUOKC-OKC,KXNBAGAME-25OCT21HOUOKC-HOU,Oklahoma City Thunder,Oklahoma City Thunder,0.705,0.305,1,_pre-game,1.010
2,0022500086,KXNBAGAME-25OCT22WASMIL-MIL,KXNBAGAME-25OCT22WASMIL-WAS,Milwaukee Bucks,Milwaukee Bucks,0.805,0.195,1,_pre-game,1.000
3,0022500003,KXNBAGAME-25OCT22CLENYK-NYK,KXNBAGAME-25OCT22CLENYK-CLE,New York Knicks,New York Knicks,0.465,0.535,1,_pre-game,1.000
4,0022500081,KXNBAGAME-25OCT22MIAORL-ORL,KXNBAGAME-25OCT22MIAORL-MIA,Orlando Magic,Orlando Magic,0.760,0.235,1,_pre-game,0.995


**Market consistency.** Each game has two markets (home wins, away wins). If prices are sensible, the two midpoints should sum to roughly 1.

In [19]:
kalshi_long["PROB_SUM"] = kalshi_long["PROB_HOME"] + kalshi_long["PROB_AWAY"]
kalshi_long.groupby("CHECKPOINT", sort=False)["PROB_SUM"].describe()[["count", "mean", "min", "max"]]

,count,mean,min,max
CHECKPOINT,,,,
_pre-game,1222.0,1.002709,0.98,1.03
EndQ1,1222.0,1.002836,0.98,1.03
EndQ2,1219.0,1.002740,0.94,1.03
EndQ3,1210.0,1.002773,0.95,1.04


**Kalshi accuracy by checkpoint.** As a game progresses, the market should become more confident and more accurate: the favorite should win more often and the Brier score should fall.

In [27]:
def market_summary(data):
    data = data.dropna(subset=["PROB_HOME"])
    return pd.Series({
        "games": len(data),
        "favorite_win_rate": ((data["PROB_HOME"] > 0.5) == data["HOME_WIN"]).mean(),
        "avg_confidence": (data["PROB_HOME"] - 0.5).abs().mean() + 0.5,
        "brier": ((data["PROB_HOME"] - data["HOME_WIN"]) ** 2).mean(),
    })

kalshi_long.groupby("CHECKPOINT", sort=False).apply(market_summary, include_groups=False).round(4)

,games,favorite_win_rate,avg_confidence,brier
CHECKPOINT,,,,
_pre-game,1222.0,0.6866,0.6974,0.1956
EndQ1,1222.0,0.6849,0.6975,0.1954
EndQ2,1220.0,0.6861,0.6975,0.1956
EndQ3,1212.0,0.6906,0.6968,0.1952


**In-game leader win rate.** How often the team leading at each checkpoint goes on to win (ties excluded). This shows how much signal the in-game stats carry and gives a benchmark to compare the Kalshi numbers against.

In [24]:
for checkpoint in IN_GAME:
    home = in_game[checkpoint][in_game[checkpoint]["MATCHUP"].str.contains("vs.")]
    not_tied = home["PLUS_MINUS"] != 0
    leader_wins = ((home["PLUS_MINUS"] > 0) == (home["WL"] == "W"))[not_tied].mean()
    print(f"{checkpoint}: leader wins {leader_wins:.1%}  ({not_tied.sum()} non-tied games)")

EndQ1: leader wins 65.6%  (1169 non-tied games)
EndQ2: leader wins 74.4%  (1196 non-tied games)
EndQ3: leader wins 82.5%  (1194 non-tied games)
